# Planet/comet encoder + trajectory decoder — extrapolation

Load the trained `PlanetEncoder` + `TrajectoryDecoder` from the latest checkpoint, pick a turn from a replay, and visualize:

1. **Board** — current positions of planets and comets, sun in the center.
2. **Predicted vs actual trajectories** — for each entity, a curve showing the decoder's 10-step extrapolation overlaid on the ground-truth path from the replay.
3. **Per-horizon error** — RMSE vs horizon, broken down by entity type (comet / orbital / static), so you can see how prediction error grows with lookahead.

If the decoder learned the dynamics well, predicted curves should sit on top of the ground-truth dotted lines for orbital planets (deterministic from `(x, y, ω)`) and reasonably close for comets (genuinely nonlinear paths).

## Setup

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'agents').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('repo:', REPO)

import gzip, json, math
import numpy as np
import torch
import matplotlib.pyplot as plt

from agents.transformer_v1.encoder.planet_encoder import PlanetEncoder
from agents.transformer_v1.encoder.pretrain_planet import TrajectoryDecoder
from agents.transformer_v1.featurizer import featurize_planets, PLANET_RAW_DIM
from agents.transformer_v1.featurizer.planet_featurizer import (
    ANCHOR_DXY_NORM, EXTRAP_HORIZONS, N_EXTRAP_HORIZONS, _is_orbiting,
)

## Load encoder + trajectory decoder

The pretrain wrapper saved `{encoder.*, heads.extrap_trajectory.net.*, ...}` into the same state dict. We strip the prefixes and load the two pieces separately.

In [ ]:
RUN_DIR = sorted((REPO / 'data' / 'encoder_runs_planet').glob('*'))[-1]
CKPT = RUN_DIR / 'planet_encoder_best.pt'
print('using:', CKPT.relative_to(REPO))

ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
d_model = ckpt['config']['d_model']

encoder = PlanetEncoder(d_model=d_model)
encoder_sd = {k.removeprefix('encoder.'): v for k, v in ckpt['model'].items()
              if k.startswith('encoder.')}
encoder.load_state_dict(encoder_sd, strict=True)
encoder.eval()

decoder = TrajectoryDecoder(d_model=d_model, n_horizons=N_EXTRAP_HORIZONS)
# heads.extrap_trajectory.net.* in checkpoint -> net.* in TrajectoryDecoder
decoder_sd = {k.removeprefix('heads.extrap_trajectory.'): v
              for k, v in ckpt['model'].items()
              if k.startswith('heads.extrap_trajectory.')}
decoder.load_state_dict(decoder_sd, strict=True)
decoder.eval()
print(f'loaded encoder (d_model={d_model}) + decoder (n_horizons={N_EXTRAP_HORIZONS}) from epoch {ckpt["epoch"]}')

## Pick a replay turn

Auto-pick a step that has comets visible — comets are where extrapolation actually matters (orbital planets are near-deterministic so the decoder essentially memorizes them).

In [ ]:
REPLAY = REPO / 'data' / 'replays' / 'Shun_PI' / '75408674_2_0.json.gz'
STEP = None  # None → auto-pick a step with comets and at least 10 future turns

with gzip.open(REPLAY, 'rt') as fh:
    replay = json.load(fh)
steps = replay['steps']

if STEP is None:
    for t, s in enumerate(steps):
        if t + max(EXTRAP_HORIZONS) >= len(steps):
            break
        if not s: continue
        ob = s[0]['observation']
        if ob and (ob.get('comet_planet_ids') or []):
            STEP = t
            break
    print(f'auto-picked STEP={STEP} (first turn with live comets and a 10-turn future)')

obs = steps[STEP][0]['observation']
n_players = len(steps[0])
print(f'replay: {REPLAY.name}  step={STEP}/{len(steps)}  '
      f'planets={len(obs["planets"])}  comets={len(obs.get("comet_planet_ids") or [])}')

## Featurize, encode, decode

In [ ]:
features, mask, records = featurize_planets(obs, learner_slot=0, num_players=n_players, max_entities=64)
n = int(mask.sum())
real_features = features[:n]                              # (n, PLANET_RAW_DIM)
with torch.no_grad():
    tokens = encoder(real_features.unsqueeze(0)).squeeze(0)   # (n, d_model)
    pred_flat = decoder(tokens)                                # (n, 2*H)
pred_dxy = pred_flat.view(n, N_EXTRAP_HORIZONS, 2).numpy() * ANCHOR_DXY_NORM   # board units
print(f'features {tuple(real_features.shape)} → tokens {tuple(tokens.shape)} → pred_dxy {pred_dxy.shape}')

## Recover ground-truth trajectories from the replay

For each entity at step `STEP`, look up its position at `STEP+1`, …, `STEP+10` directly from the replay.

In [ ]:
from agents.transformer_v1.featurizer.planet_featurizer import _planet_pos_at

actual_xy = np.full((n, N_EXTRAP_HORIZONS, 2), np.nan)
for i, rec in enumerate(records):
    for j, h in enumerate(EXTRAP_HORIZONS):
        fut = _planet_pos_at(steps, STEP + h, rec.planet_id)
        if fut is not None:
            actual_xy[i, j] = fut

# Predicted absolute positions = current pos + predicted dx/dy
pred_xy = np.empty_like(actual_xy)
for i, rec in enumerate(records):
    pred_xy[i, :, 0] = rec.x + pred_dxy[i, :, 0]
    pred_xy[i, :, 1] = rec.y + pred_dxy[i, :, 1]

# Per-entity classification for nicer plotting
av = float(obs.get('angular_velocity') or 0.0)
kinds = []
for rec in records:
    if rec.is_comet:
        kinds.append('comet')
    elif _is_orbiting(rec.x, rec.y, rec.radius, av):
        kinds.append('orbital')
    else:
        kinds.append('static')
from collections import Counter
print('entity types:', Counter(kinds))

## Board view with predicted vs actual trajectories

Solid line = decoder prediction, dashed = ground truth. Sun in yellow. Static planets are dimmed since they sit still.

In [ ]:
OWNER_COLORS = {0: '#1f77b4', 1: '#d62728', 2: '#2ca02c', 3: '#9467bd', -1: '#888888'}
KIND_PRED_COLOR = {'comet': '#ff7f0e', 'orbital': '#1f77b4', 'static': '#666666'}
KIND_ALPHA = {'comet': 0.95, 'orbital': 0.8, 'static': 0.25}

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect('equal')
ax.set_facecolor('#0b1020')
ax.add_patch(plt.Circle((50, 50), 10, color='#ffd700', alpha=0.85, zorder=1))

for i, rec in enumerate(records):
    kind = kinds[i]
    color = KIND_PRED_COLOR[kind]
    alpha = KIND_ALPHA[kind]

    # current position marker (sized by ships)
    ms = max(40, math.sqrt(max(1, rec.ships)) * 8)
    edge = '#ffffff' if kind == 'comet' else '#222244'
    ax.scatter([rec.x], [rec.y], c=OWNER_COLORS.get(rec.owner_id, '#888'),
               s=ms, ec=edge, lw=0.8, zorder=4, alpha=min(1.0, alpha + 0.1))

    # predicted trajectory: include current pos at horizon 0
    px = np.concatenate([[rec.x], pred_xy[i, :, 0]])
    py = np.concatenate([[rec.y], pred_xy[i, :, 1]])
    ax.plot(px, py, '-', color=color, lw=1.4, alpha=alpha, zorder=3,
            label=f'pred ({kind})' if i == [j for j, k in enumerate(kinds) if k == kind][0] else None)

    # actual trajectory (where we have it)
    ax_pts = actual_xy[i]
    valid = ~np.isnan(ax_pts).any(axis=1)
    if valid.any():
        gx = np.concatenate([[rec.x], ax_pts[valid, 0]])
        gy = np.concatenate([[rec.y], ax_pts[valid, 1]])
        ax.plot(gx, gy, '--', color=color, lw=1.0, alpha=alpha * 0.6, zorder=2)

ax.set_title(f'{REPLAY.name}  step {STEP} → {STEP + max(EXTRAP_HORIZONS)} '
             f'(solid = predicted, dashed = actual)', color='white')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_color('white')
# Build a clean legend manually since we labeled only the first of each kind
handles = [plt.Line2D([0], [0], color=KIND_PRED_COLOR[k], lw=2, label=k)
           for k in ('comet', 'orbital', 'static')]
handles.append(plt.Line2D([0], [0], color='white', lw=1.0, ls='--', label='ground truth'))
ax.legend(handles=handles, loc='upper right', facecolor='#11183a', labelcolor='white', fontsize=9)
plt.tight_layout(); plt.show()

## Per-horizon error breakdown

RMSE vs horizon, separated by entity type. Static planets should be ~0 (no motion). Orbital planets should have very low error (linear in encoded angle). Comets are the interesting case — error grows with horizon as the decoder's path estimate drifts.

In [ ]:
kinds_arr = np.array(kinds)
errs = np.linalg.norm(pred_xy - actual_xy, axis=-1)   # (n, H), Euclidean distance
valid = ~np.isnan(errs)

fig, ax = plt.subplots(figsize=(8, 5))
for kind in ('static', 'orbital', 'comet'):
    sel = (kinds_arr == kind)
    if not sel.any():
        continue
    rmse = np.sqrt(np.nanmean(errs[sel] ** 2, axis=0))
    ax.plot(EXTRAP_HORIZONS, rmse, marker='o',
            color=KIND_PRED_COLOR[kind], label=f'{kind} (n={int(sel.sum())})')
ax.set_xlabel('horizon (turns ahead)')
ax.set_ylabel('RMSE (board units)')
ax.set_title('extrapolation error vs horizon, by entity type')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()

# Also print numbers
print('\nper-horizon RMSE:')
for kind in ('static', 'orbital', 'comet'):
    sel = (kinds_arr == kind)
    if not sel.any():
        continue
    rmse = np.sqrt(np.nanmean(errs[sel] ** 2, axis=0))
    print(f'  {kind:<8s}  ' + '  '.join(f'h{h}={r:.3f}' for h, r in zip(EXTRAP_HORIZONS, rmse)))

## Optional — pick a single comet and inspect its forecast closely

In [ ]:
comet_idx = [i for i, k in enumerate(kinds) if k == 'comet']
if comet_idx:
    i = comet_idx[0]
    rec = records[i]
    print(f'comet planet_id={rec.planet_id} at ({rec.x:.2f}, {rec.y:.2f})')
    print(f'{"h":>3s}  {"pred (x, y)":>22s}  {"actual (x, y)":>22s}  {"err":>6s}')
    for j, h in enumerate(EXTRAP_HORIZONS):
        px, py = pred_xy[i, j]
        ax_, ay_ = actual_xy[i, j]
        err = math.hypot(px - ax_, py - ay_) if not np.isnan(ax_) else float('nan')
        actual_str = f'({ax_:.2f}, {ay_:.2f})' if not np.isnan(ax_) else '(missing)'
        print(f'{h:>3d}  ({px:>7.2f}, {py:>7.2f})  {actual_str:>22s}  {err:>6.3f}')
else:
    print('no comets in this step')